# Tome 7, Chapter 6: Distributions and Sobolev Spaces (Python companion)

Exercises first, solutions at the end. Distributions on a computer: a finite element solver, the boundary layer that breaks it, edge detection as differentiation of distributions, and a function in $H^{1/2}$ that is infinite only on paper. Requirements: numpy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P6.1 (★★) CS application: a finite element solver.** Write a finite element solver with hat functions (Figure 6.2) for $-u''+u=f$ on $]0,1[$, $u(0)=u(1)=0$: assemble the tridiagonal stiffness and mass matrices, compute the load vector by Gaussian quadrature, and solve. Test it on $f=(\pi^2+1)\sin\pi x$, whose solution is $\sin\pi x$. Measure the error in $L^2$ and in $H^1$ for $N=8,16,\dots,512$ elements and read off the orders. Then check Céa's lemma (Proposition 6.19): compare the error of $u_h$ in the energy norm with that of the piecewise linear interpolant of $u$. Which is smaller, and why must it be?

**P6.2 (★★) The boundary layer.** Solve $-u''+bu'+u=b$, $u(0)=u(1)=0$ (Exercise 6.6, with $f=b$) exactly and by finite elements, for $b=100$, with $N=20$ and $N=200$ elements. Plot both. What goes wrong on the coarse mesh? Relate it to the *mesh Péclet number* $\frac{bh}2$. Try the classical cure, adding an artificial diffusion $\frac{bh}2$ to the equation: what does it fix, and what does it cost?

**P6.3 (★) CS application: edges are Dirac masses.** A signal on $[0,1]$ jumps up by $1$ at $x=0.3$ and down by $1$ at $x=0.7$, and is sampled at $2000$ points with Gaussian noise of standard deviation $0.1$. Its distributional derivative is $\delta_{0.3}-\delta_{0.7}$ (Example 6.5). Show that finite differences of the noisy samples are fragile, but that convolving with $\varphi_t'$, the derivative of a mollifier, which by Proposition 6.7 computes $(T*\varphi_t)'=T'*\varphi_t$, locates both edges. How does the choice of $t$ trade noise against precision? (This is the idea behind the Canny edge detector.)

**P6.4 (★) Unbounded, but only just.** Let $g(\xi)=\frac1{(1+|\xi|)\log(2+|\xi|)}$ as in Exercise 6.3. Compute $f*G_\varepsilon(0)=\int g(\xi)e^{-\pi\varepsilon^2\xi^2}\dd\xi$ for $\varepsilon=10^{-1},\dots,10^{-12}$, and the $H^{1/2}$ norm $\int(1+\xi^2)^{1/2}g^2$. How large is the ``infinite'' value $f(0)$ at any scale you will ever see? (Substitute $\xi=e^s$ to integrate over huge ranges.)

## Solutions

### P6.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

gx, gw = np.polynomial.legendre.leggauss(5)   # Gauss points on [-1,1]

def fem(N, f):
    h = 1.0 / N
    n = N - 1                                  # interior nodes
    K = (np.diag(2 * np.ones(n)) - np.diag(np.ones(n - 1), 1) - np.diag(np.ones(n - 1), -1)) / h
    M = (np.diag(4 * np.ones(n)) + np.diag(np.ones(n - 1), 1) + np.diag(np.ones(n - 1), -1)) * h / 6
    F = np.zeros(N + 1)
    for e in range(N):                         # load vector, element by element
        x = e * h + (gx + 1) * h / 2
        w = gw * h / 2
        F[e] += np.sum(w * f(x) * (e * h + h - x) / h)
        F[e + 1] += np.sum(w * f(x) * (x - e * h) / h)
    U = np.zeros(N + 1)
    U[1:-1] = np.linalg.solve(K + M, F[1:-1])
    return U

def errors(U, u, du):
    N = len(U) - 1; h = 1.0 / N
    e0 = e1 = 0.0
    for e in range(N):
        x = e * h + (gx + 1) * h / 2
        w = gw * h / 2
        uh = U[e] + (U[e + 1] - U[e]) * (x - e * h) / h
        duh = (U[e + 1] - U[e]) / h
        e0 += np.sum(w * (u(x) - uh) ** 2)
        e1 += np.sum(w * (du(x) - duh) ** 2)
    return np.sqrt(e0), np.sqrt(e1)

u = lambda x: np.sin(np.pi * x)
du = lambda x: np.pi * np.cos(np.pi * x)
f = lambda x: (np.pi ** 2 + 1) * np.sin(np.pi * x)
print(" N     L2 error     H1 error    energy(FEM)  energy(interp)")
prev = None
for N in [8, 16, 32, 64, 128, 256, 512]:
    U = fem(N, f)
    l2, h1 = errors(U, u, du)
    I = u(np.linspace(0, 1, N + 1))            # the interpolant
    il2, ih1 = errors(I, u, du)
    print(f"{N:4d}  {l2:.3e}  {h1:.3e}  {np.hypot(l2, h1):.6e}  {np.hypot(il2, ih1):.6e}")
    if prev: print(f"      ratios: L2 {prev[0]/l2:.2f}, H1 {prev[1]/h1:.2f}")
    prev = (l2, h1)

The $L^2$ error is divided by $4$ and the $H^1$ error by $2$ each time $h$ is halved: orders $h^2$ and $h$. The $H^1$ order is Céa's lemma plus the interpolation estimate $\|u-I_hu\|_{H^1}\le Ch\|u''\|_2$; the extra power in $L^2$ comes from a duality trick (Aubin–Nitsche) that the exercise list of a finite element course always contains. The last two columns are the energy norms $\sqrt{\int(v'^2+v^2)}$ of $u-u_h$ and $u-I_hu$. The finite element error is always the smaller of the two, slightly: since $a$ is symmetric here, $u_h$ is *exactly* the best approximation of $u$ from $V_h$ in the energy norm (the Galerkin orthogonality of Proposition 6.19 says $u-u_h\perp V_h$ for the inner product $a$), and the interpolant is just one competitor. That is the whole philosophy of the method: you never compute the best approximation of the unknown $u$ directly, yet Galerkin finds it for you.

### P6.2

In [ ]:
def exact(b, x):
    s = np.sqrt(b * b + 4)
    r1, r2 = (b - s) / 2, (b + s) / 2
    # u = b + A e^{r1 x} + C e^{r2 (x-1)}, written to avoid overflow
    A, C = np.linalg.solve([[1, np.exp(-r2)], [np.exp(r1), 1]], [-b, -b])
    return b + A * np.exp(r1 * x) + C * np.exp(r2 * (x - 1))

def fem_cd(N, b, extra=0.0):
    h = 1.0 / N; n = N - 1
    eps = 1.0 + extra
    T = lambda d, o: np.diag(d * np.ones(n)) + np.diag(o[0] * np.ones(n - 1), 1) + np.diag(o[1] * np.ones(n - 1), -1)
    A = eps * T(2, (-1, -1)) / h + b * T(0, (0.5, -0.5)) + T(4, (1, 1)) * h / 6
    U = np.zeros(N + 1)
    U[1:-1] = np.linalg.solve(A, b * h * np.ones(n))
    return U

b = 100
xx = np.linspace(0, 1, 2001)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for k, (N, extra, title) in enumerate([(20, 0, "N=20"), (200, 0, "N=200"), (20, b / 2 / 20, "N=20, artificial diffusion")]):
    U = fem_cd(N, b, extra)
    x = np.linspace(0, 1, N + 1)
    ax[k].plot(xx, exact(b, xx), 'k--', lw=1, label='exact')
    ax[k].plot(x, U, 'o-', ms=3, label='FEM')
    ax[k].set_title(title); ax[k].legend()
    err = np.max(np.abs(U - exact(b, x)))
    print(f"{title:28s} Peclet {b / (2 * N):5.2f}  min U {U[1:-1].min():+.3f}  max U {U.max():.3f}  nodal error {err:.3f}")
plt.show()

The exact solution is close to $x$ and drops to $0$ in a layer of width about $\frac1b=0.01$ at the outflow end, as Exercise 6.6 predicted. On the coarse mesh ($h=0.05$, five times the layer width, Péclet number $2.5$) the finite element solution is excellent far from the layer and then zigzags as it approaches it: the last nodal values read $0.76, 0.93, 0.71, 1.37$, overshooting far above the true maximum $0.94$, the wiggles being largest next to the layer and decaying upstream. Galerkin is not *unstable* in the sense of Lax–Milgram, which still gives a unique solution with Céa's bound, but the constant $\frac M\alpha=1+b=101$ is huge, and the best approximation from $V_h$ of a function with an unresolved layer is poor in $H^1$. On the fine mesh (Péclet number $0.25$) everything is fine. Adding the artificial diffusion $\frac{bh}2$ (the "upwind" scheme in disguise) kills the oscillations, but it solves a different equation, with diffusion $3.5$ instead of $1$: the solution is now monotone up to the layer, but too low there ($0.87$ and $0.78$ instead of $0.90$ and $0.94$), the layer having been smeared over several elements. Modern methods (streamline-upwind Petrov–Galerkin, discontinuous Galerkin) add diffusion only along the flow, and the whole field of convection-dominated problems, from weather to air flowing past a wing, grew out of this small picture.

### P6.3

In [ ]:
rng = np.random.default_rng(0)
n = 2000
x = (np.arange(n) + 0.5) / n
h = 1.0 / n
signal = ((x > 0.3) & (x < 0.7)).astype(float)
noisy = signal + 0.1 * rng.standard_normal(n)

fd = np.diff(noisy) / h
away = (np.abs(x[:-1] - 0.3) > 0.01) & (np.abs(x[:-1] - 0.7) > 0.01)
print("finite differences: at the edges", round(fd[np.argmin(np.abs(x[:-1] - 0.3))]), round(fd[np.argmin(np.abs(x[:-1] - 0.7))]),
      "| away from them: std", round(np.std(fd[away])), " max", round(np.abs(fd[away]).max()))

def dmollifier(t):
    s = np.arange(-int(t / h), int(t / h) + 1) * h
    u = s / t
    phi = np.where(np.abs(u) < 1, np.exp(-1 / np.maximum(1 - u ** 2, 1e-12)), 0.0)
    dphi = np.where(np.abs(u) < 1, phi * (-2 * u / np.maximum(1 - u ** 2, 1e-12) ** 2) / t, 0.0)
    c = 1 / (phi.sum() * h)
    return c * dphi

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(x, noisy, lw=0.5); ax[0].set_title("noisy signal")
for t in [0.005, 0.02, 0.08]:
    D = np.convolve(noisy, dmollifier(t), mode='same') * h
    i1 = np.argmax(D[: n // 2]); i2 = n // 2 + np.argmin(D[n // 2:])
    print(f"t = {t:5.3f}: edges found at {x[i1]:.4f} and {x[i2]:.4f}, peak heights {D[i1]:.1f} and {D[i2]:.1f}")
    ax[1].plot(x, D, lw=1, label=f"t={t}")
ax[1].legend(); ax[1].set_title(r"$T*\varphi_t'$")
plt.show()

The finite differences of the noisy samples are about $\pm2000$ at the edges (a jump of $1$ divided by $h=0.0005$), but away from them the noise alone gives slopes with standard deviation about $285$ (that is $0.1\sqrt2/h$) and a maximum near $900$. The edges still win here, but only by a factor of two: with noise $0.25$ they would drown. Differentiation amplifies noise, which is the analyst's way of saying that $\frac{\dd}{\dd x}$ is unbounded. Convolving with $\varphi_t'$ instead moves the derivative onto the smooth test function, exactly as in the definition of a distributional derivative, and the result is $\delta_{0.3}*\varphi_t-\delta_{0.7}*\varphi_t$ plus a little smoothed noise: two bumps of heights about $\varphi_t(0)=\frac{0.83}t$ (that is $166$, $41$ and $10$; the measured peaks agree up to noise), at the edges. A small $t$ gives tall, sharp bumps but more noise; a large $t$ gives clean but wide bumps; the edge positions are found to within a few samples in all three cases. Canny's edge detector (1986) does exactly this in two dimensions with a Gaussian in place of the bump, and his paper derives the trade-off you just saw as an optimisation problem.

### P6.4

In [ ]:
from math import pi, exp, log
g = lambda xi: 1 / ((1 + xi) * np.log(2 + xi))
s = np.linspace(-30, 80, 400001)              # xi = e^s, from 1e-13 to 5e34
xi = np.exp(s)
ds = s[1] - s[0]
print(" epsilon     f*G_eps(0)")
for k in range(1, 13):
    eps = 10.0 ** (-k)
    val = 2 * np.sum(g(xi) * np.exp(-pi * eps ** 2 * xi ** 2) * xi) * ds
    print(f"  1e-{k:02d}    {val:.4f}")
for R in [1e2, 1e6, 1e12, 1e34]:
    m = xi < R
    tail = 2 * np.sum(np.sqrt(1 + xi[m] ** 2) * g(xi[m]) ** 2 * xi[m]) * ds
    print(f"H^(1/2) integral up to {R:.0e}: {tail:.4f}")

The smoothed values $f*G_\varepsilon(0)$ do increase without bound, as Exercise 6.3 proved, but at the speed of $2\log\log\frac1\varepsilon$: each factor $10$ in $\varepsilon$ adds only a fraction of a unit, and at $\varepsilon=10^{-12}$ the value is still below $8.5$. Meanwhile the $H^{1/2}$ integral converges, to about $3.34$, but slowly: its tail beyond $R$ is about $2\int_R^\infty\frac{\dd\xi}{\xi\log^2\xi}=\frac2{\log R}$, still $0.03$ at $R=10^{34}$, which is why the printed partial values are still creeping up. So $f$ is an honest element of $H^{1/2}$ that is ``infinite at $0$'' only in a sense no measurement could ever detect. This is typical of borderline cases in analysis: the Sobolev embedding fails at $s=\frac12$ by a doubly logarithmic margin, and what replaces it is the space BMO of functions of bounded mean oscillation, or the Moser–Trudinger inequality, which says that $e^{cf^2}$ is still integrable.